# The value of $\Gamma^{\text{wind}}$ cannot be found by big-M heuristic
This notebook demonstrates that on the synthetic data instances, the big-M
heuristic fails to terminate before excessively large values of $\Gamma^{\text{wind}}$.

In [ ]:
# IMPORTS
# -------
import wind_data_module as data
import wind_model as model

import importlib
import pandas as pd

# Optimizer
import xpress as xp
# N.B. A licence is required for xpress.
# xp.init('C:/Program Files/xpressmp/bin/xpauth_personal.xpr') # for local version
xp.init('C:/Program Files/xpressmp/bin/xpauth_uoe2.xpr') # For UoE license

In [9]:
importlib.reload(data)
importlib.reload(model)

<module 'wind_model_04' from 'c:\\Users\\ajmac\\Documents\\ORMSc\\Code\\Required\\wind_model_04.py'>

In [10]:
# stc = {'line_data': "000line.csv",
#        'bid': "000bid.csv",
#        'offer': "000offer.csv",
#        'wind': "000wind.csv"}

In [11]:
data_params = {"kappa": 10, "nb_scenarios":5, "c_CO2":0.225, 
               "Gamma_wind":100_000,
               "c_max": 300_000_000}

In [12]:
data.simulate_data(10,3, parameters=data_params)

In [13]:
bilevel = True
new_lines = []
model_params = {"bilevel": bilevel, "new_lines": new_lines}

sol_dict = model.run_model(data.DataStore, solver_options={"outputlog":0}, wind=True)

In [14]:
# Largest value of phi_minus
phi_minus_max = max([abs(x) for x in sol_dict['phi_minus'].values()])

# Largest value of the other quantity that should be bounded by Gamma_max
phi = sol_dict['phi']
phi_minus = sol_dict['phi_minus']
quant = [phi[k] - phi_minus[k] for k in sol_dict['phi'].keys()]
quant_max = max([abs(x) for x in quant])

print(f"Gamma_max is set at {data.Gamma_max}.")
print(f"The quantities which should be bound by this are {phi_minus_max} and {quant_max}.")


# Largest value of varsigma
varsigma_max = max([abs(x) for x in sol_dict['varsigma'].values()])

# Largest value of the other quantity that should be bounded by Gamma_wind:
gamma = sol_dict['gamma']
varsigma = sol_dict['varsigma']
quant_2 = [gamma[k] - varsigma[k] for k in sol_dict['gamma'].keys()]
quant_max_2 = max([abs(x) for x in quant])

print("")
print(f"Gamma_wind is set at {data.Gamma_wind}.")
print(f"The largest value of quantities which should be below this is  {max(varsigma_max, quant_max_2)}.")

# Identfy if wind power projects are opened:
opened = False
for k, v in sol_dict["x_P"].items():
    if v>0.5:
        print(f"{k} is opened.")
        opened = True

if not opened:
    print("")
    print("No wind power projects opened.")

Gamma_max is set at 100000.
The quantities which should be bound by this are 100000.00000000023 and 100000.0.

Gamma_wind is set at 100000.
The largest value of quantities which should be below this is  100000.0.

No wind power projects opened.
